# 09 - GPU batches and parameter sweeps

`GPULattice` packs supported elements for one fused lattice kernel per pass. Particle matrices are `N x 6`. The CPU backend uses exactly the same packed representation.

In [ ]:
import Pkg
EXAMPLES_DIR = isfile(joinpath(pwd(), "common.jl")) ? pwd() : joinpath(pwd(), "examples")
Pkg.activate(EXAMPLES_DIR)
using TrackPad
include(joinpath(EXAMPLES_DIR, "common.jl")); using .TrackPadExamples
ring, beam = madx_fodo()

In [ ]:
N = 100_000
coordinates = zeros(Float64, N, 6)
coordinates[:, 1] .= range(-1e-3, 1e-3; length=N)
gl_cpu = GPULattice(ring, beam; dtype=Float64)
batch_ringpass!(coordinates, gl_cpu, 10)
extrema(coordinates[:, 1])

In [ ]:
qf_index = findfirst(e -> e.name == :qfh, ring.elements)
qd_index = findfirst(e -> e.name == :qd, ring.elements)
qf_values = collect(range(1.2, 1.6; length=21))
qd_values = collect(range(-1.6, -1.2; length=17))
sweep = ParamSweepLattice(
    gl_cpu, [(qf_index, 2, qf_values), (qd_index, 2, qd_values)]; mode=:cartesian,
)
inputs = repeat(reshape([1e-3, 0, 0, 0, 0, 0], 1, 6), Int(sweep.n_configs), 1)
outputs = similar(inputs)
param_sweep_linepass!(outputs, inputs, sweep)

(configurations=Int(sweep.n_configs), storage=size(sweep.variation_values),
 output_x_range=extrema(outputs[:, 1]))

For aligned Monte Carlo trials, pass equal-length error vectors and leave `mode=:aligned`. For an NVIDIA GPU, activate `environments/cuda`, then move the packed objects and arrays:

In [ ]:
# using CUDA
# gl_gpu = gpu_adapt(gl_cpu, CUDA.CUDABackend())
# sweep_gpu = gpu_adapt(sweep, CUDA.CUDABackend())
# coordinates_gpu = CuArray(coordinates)
# batch_ringpass!(coordinates_gpu, gl_gpu, 10)
# outputs_gpu = CUDA.zeros(Float64, Int(sweep_gpu.n_configs), 6)
# param_sweep_linepass!(outputs_gpu, CuArray(inputs), sweep_gpu)

Unsupported elements or settings throw while constructing `GPULattice`; TrackPad never substitutes a drift. CUDA supports `Float32` and `Float64`; Apple Metal uses `Float32`. Compilation and host-device transfers should be excluded from device-resident benchmarks.